# Discovery Chain Generation
Generates 500 biomedical reasoning chains (250 MedQA + 250 BioASQ) via OpenRouter.  
Chains are saved to `data/discovery_chains.jsonl` with checkpointing after every 50.  
No UMLS scoring — raw chain text only for qualitative discovery analysis.

In [ ]:
import os, sys, json, time, random
from pathlib import Path

PROJECT_ROOT = str(Path.cwd())
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# Set API key BEFORE importing cot_generator (it reads env at import time)
OPENROUTER_API_KEY = input("OpenRouter API key: ").strip()
os.environ["OPENROUTER_API_KEY"] = OPENROUTER_API_KEY

MODEL = "anthropic/claude-haiku-4-5"
N_PER_SOURCE = 250
CHECKPOINT_EVERY = 50
OUT_JSONL = Path(PROJECT_ROOT) / "data" / "discovery_chains.jsonl"
OUT_JSON  = Path(PROJECT_ROOT) / "data" / "discovery_chains.json"
OUT_JSONL.parent.mkdir(exist_ok=True)

print(f"Output: {OUT_JSONL}")
print(f"Model:  {MODEL}")
print(f"Target: {N_PER_SOURCE * 2} chains ({N_PER_SOURCE} per source)")

In [ ]:
import utils.cot_generator as _cg

# Patch module-level key in case it was cached before the env var was set
_cg.OPENROUTER_API_KEY = OPENROUTER_API_KEY
_cg.OPENROUTER_READY   = bool(OPENROUTER_API_KEY)

from utils.cot_generator import generate
print(f"cot_generator loaded. OPENROUTER_READY={_cg.OPENROUTER_READY}")

In [ ]:
from datasets import load_dataset

medqa_questions = []

# bigbio/med_qa uses a deprecated loading script — use GBaker/MedQA-USMLE-4-options instead
medqa_candidates = [
    ("GBaker/MedQA-USMLE-4-options", "test"),
    ("GBaker/MedQA-USMLE-4-options", "train"),
    ("medalpaca/medical_meadow_medqa", "train"),
]

for ds_name, split in medqa_candidates:
    try:
        ds = load_dataset(ds_name, split=split)
        for i, row in enumerate(ds):
            # GBaker: question (str), options (dict A/B/C/D), answer_idx (int), answer (str)
            # medalpaca: instruction (str), input (str), output (str)
            q_text = (row.get("question") or row.get("instruction") or "").strip()
            opts = row.get("options", {})
            if isinstance(opts, dict) and opts:
                options_str = "  ".join(f"{k}: {v}" for k, v in opts.items())
                full_q = f"{q_text}\nOptions: {options_str}"
            else:
                full_q = q_text
            answer_text = str(row.get("answer") or row.get("output") or "")
            answer_idx  = str(row.get("answer_idx", ""))
            if q_text:
                medqa_questions.append({
                    "id": f"medqa_{i:04d}",
                    "source": "medqa",
                    "question": full_q,
                    "question_text": q_text,
                    "options": opts if isinstance(opts, dict) else {},
                    "answer": answer_idx,
                    "answer_text": answer_text,
                })
        print(f"MedQA loaded from '{ds_name}' ({split}): {len(medqa_questions)} questions")
        break
    except Exception as e:
        print(f"  '{ds_name}' ({split}) failed: {e}")

if not medqa_questions:
    print("WARNING: No MedQA questions loaded.")

random.seed(42)
random.shuffle(medqa_questions)
medqa_sample = medqa_questions[:N_PER_SOURCE]
print(f"Sampled {len(medqa_sample)} MedQA questions")

In [ ]:
bioasq_questions = []

# enelpol/rag-mini-bioasq requires config name 'question-answer-passages'
bioasq_candidates = [
    ("enelpol/rag-mini-bioasq", "question-answer-passages", "train"),
    ("enelpol/rag-mini-bioasq", "question-answer-passages", "test"),
]

for ds_name, config, split in bioasq_candidates:
    try:
        ds = load_dataset(ds_name, config, split=split)
        for i, row in enumerate(ds):
            q = (row.get("question") or row.get("body") or "").strip()
            a = row.get("answer") or row.get("ideal_answer") or ""
            if isinstance(a, list):
                a = a[0] if a else ""
            if q:
                bioasq_questions.append({
                    "id": f"bioasq_{i:04d}",
                    "source": "bioasq",
                    "question": q,
                    "question_text": q,
                    "options": {},
                    "answer": "",
                    "answer_text": str(a),
                })
        print(f"BioASQ loaded from '{ds_name}' ({config}, {split}): {len(bioasq_questions)} questions")
        break
    except Exception as e:
        print(f"  '{ds_name}' ({config}, {split}) failed: {e}")

if not bioasq_questions:
    print("WARNING: No BioASQ questions loaded — will use MedQA only.")

random.seed(43)
random.shuffle(bioasq_questions)
bioasq_sample = bioasq_questions[:N_PER_SOURCE]
print(f"Sampled {len(bioasq_sample)} BioASQ questions")

In [ ]:
all_questions = medqa_sample + bioasq_sample
random.seed(99)
random.shuffle(all_questions)

print(f"Total questions to process: {len(all_questions)}")
print(f"  MedQA:  {sum(1 for q in all_questions if q['source'] == 'medqa')}")
print(f"  BioASQ: {sum(1 for q in all_questions if q['source'] == 'bioasq')}")

# Check for existing checkpoint to resume
done_ids = set()
if OUT_JSONL.exists():
    with open(OUT_JSONL) as f:
        for line in f:
            try:
                rec = json.loads(line)
                done_ids.add(rec["id"])
            except Exception:
                pass
    print(f"Resuming: {len(done_ids)} chains already generated")
else:
    print("Starting fresh — no checkpoint found")

In [ ]:
# DEBUG: step through _call_openrouter manually to find the real error
import traceback

print(f"OPENROUTER_API_KEY set: {bool(_cg.OPENROUTER_API_KEY)}")
print(f"Key prefix: {_cg.OPENROUTER_API_KEY[:8]}...")

try:
    from openai import OpenAI
    print("openai import: OK")
except Exception:
    print("openai import FAILED:"); traceback.print_exc()

try:
    client = OpenAI(api_key=_cg.OPENROUTER_API_KEY, base_url=_cg.OPENROUTER_BASE_URL)
    print("OpenAI client created: OK")
except Exception:
    print("OpenAI client FAILED:"); traceback.print_exc()

try:
    resp = client.chat.completions.create(
        model=MODEL,
        temperature=0.2,
        max_tokens=700,
        messages=[
            {"role": "system", "content": "Return concise numbered reasoning steps only."},
            {"role": "user",   "content": "What is aspirin used for? Break into steps."},
        ],
        extra_headers={
            "HTTP-Referer": "https://github.com/biomedical-semantic-leakage",
            "X-Title": "Biomedical Semantic Leakage Detection",
        },
    )
    print("API call OK:", resp.choices[0].message.content[:100])
except Exception:
    print("API call FAILED:"); traceback.print_exc()

In [ ]:
import traceback

pending = [q for q in all_questions if q["id"] not in done_ids]
print(f"Generating {len(pending)} remaining chains...")

failures = []
generated_this_run = 0

with open(OUT_JSONL, "a", encoding="utf-8") as f_out:
    for i, q_meta in enumerate(pending):
        try:
            result = generate(q_meta["question"], prefer="openrouter", model=MODEL)
            steps = result.get("steps", [])

            if not steps or result.get("provider") == "local":
                failures.append({"id": q_meta["id"], "reason": "fallback or empty steps"})
                print(f"  [{i+1}/{len(pending)}] SKIP {q_meta['id']} — fallback/empty")
                continue

            record = {
                **q_meta,
                "steps": steps,
                "n_steps": len(steps),
                "model": result.get("model", MODEL),
                "provider": result.get("provider", "openrouter"),
                "generated_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
            }
            f_out.write(json.dumps(record, ensure_ascii=False) + "\n")
            f_out.flush()
            generated_this_run += 1

            if (i + 1) % CHECKPOINT_EVERY == 0:
                print(f"  [{i+1}/{len(pending)}] checkpoint — {generated_this_run} generated, {len(failures)} failures")
            else:
                print(f"  [{i+1}/{len(pending)}] {q_meta['id']} ({q_meta['source']}) — {len(steps)} steps", end="\r")

        except KeyboardInterrupt:
            print("\nInterrupted — progress saved to checkpoint.")
            break
        except Exception as e:
            failures.append({"id": q_meta["id"], "reason": str(e)})
            print(f"\n  [{i+1}/{len(pending)}] ERROR {q_meta['id']}: {e}")
            time.sleep(2)  # brief pause on error before continuing

print(f"\nDone. Generated {generated_this_run} chains this run, {len(failures)} failures.")

In [ ]:
# Consolidate JSONL -> single JSON array for easy reading
all_chains = []
with open(OUT_JSONL, encoding="utf-8") as f:
    for line in f:
        try:
            all_chains.append(json.loads(line))
        except Exception:
            pass

with open(OUT_JSON, "w", encoding="utf-8") as f:
    json.dump(all_chains, f, indent=2, ensure_ascii=False)

print(f"Consolidated {len(all_chains)} chains -> {OUT_JSON}")

In [ ]:
import statistics

by_source = {}
step_counts = []
for c in all_chains:
    src = c.get("source", "unknown")
    by_source[src] = by_source.get(src, 0) + 1
    step_counts.append(c.get("n_steps", len(c.get("steps", []))))

print("=" * 50)
print(f"SUMMARY")
print("=" * 50)
print(f"Total chains:     {len(all_chains)}")
for src, cnt in sorted(by_source.items()):
    print(f"  {src:10s}:  {cnt}")
if step_counts:
    print(f"Steps per chain:  mean={statistics.mean(step_counts):.1f}  "
          f"median={statistics.median(step_counts):.0f}  "
          f"min={min(step_counts)}  max={max(step_counts)}")
print(f"Failures:         {len(failures)}")
if failures:
    for fl in failures[:10]:
        print(f"  {fl['id']}: {fl['reason']}")
print("=" * 50)
print(f"Output files:")
print(f"  {OUT_JSONL}  ({OUT_JSONL.stat().st_size // 1024} KB)")
print(f"  {OUT_JSON}   ({OUT_JSON.stat().st_size // 1024} KB)")